# Lahore Air Quality & Weather - ETL + EDA

Run the pipeline first (`python run_all.py`). This notebook **reads the data from the database** (the result of the Extract, Transform/Validate and Load stages) and explores it. Every chart is also saved in `outputs/`.

## 0. What the pipeline did
Row counts at each step (from `processed/transform_report.json`):

In [ ]:
import json
import pandas as pd
import eda
import config

report = json.load(open(config.PROCESSED_DIR / 'transform_report.json'))
pd.Series(report, name='rows').to_frame()

## 1. Reading the data back from the database
Here is a SQL query run directly on the SQLite database, then the full table loaded for analysis.

In [ ]:
import sqlite3
conn = sqlite3.connect(config.DB_PATH)
sql = """
SELECT strftime('%Y-%m', time) AS month,
       ROUND(AVG(pm2_5), 1)  AS mean_pm25,
       ROUND(MAX(pm2_5), 1)  AS max_pm25,
       COUNT(*)              AS hours
FROM air_weather
GROUP BY month
ORDER BY month
"""
result = pd.read_sql_query(sql, conn)
conn.close()
result

In [ ]:
df = eda.load_data()
daily = eda.daily_table(df)
print(len(df), 'hourly rows from', df['time'].min(), 'to', df['time'].max())
df.head()

## 2. Summary statistics and missing values

In [ ]:
stats, miss = eda.save_tables(df, daily)
stats

In [ ]:
miss

## 3. Pollutant distributions

In [ ]:
eda.fig_distribution(df);

In [ ]:
eda.fig_pollutants(df);

## 4. Patterns: hour of day, day by day, month by month

In [ ]:
eda.fig_hourly(df);

In [ ]:
eda.fig_daily(daily);

In [ ]:
eda.fig_monthly(df);

## 5. Weather vs pollution (correlation)
Values near +1 or -1 mean a strong relationship, near 0 means almost none. Correlation shows association, not cause.

In [ ]:
eda.fig_corr(df);

In [ ]:
eda.fig_scatter(df);

## 6. Days above the WHO guideline

In [ ]:
eda.fig_who(daily);

In [ ]:
daily.sort_values('pm2_5_mean', ascending=False).head(10)

## 7. Findings (numbers computed from the data)

In [ ]:
from IPython.display import Markdown
Markdown(eda.write_findings(df, daily))